In [19]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col,when,dense_rank,row_number,rank
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,LongType
from pyspark.sql.window import Window

In [2]:
spark=SparkSession.builder.appName("LeetcodeSQL50").getOrCreate()

26/07/07 22:03:33 WARN Utils: Your hostname, Chinmayas-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.7 instead (on interface en0)
26/07/07 22:03:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/07 22:03:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.version

'3.5.7'

## 1757. Recyclable and Low Fat Products
''' 

Example 1:

Input: 
Products table:
+-------------+----------+------------+
| product_id  | low_fats | recyclable |
+-------------+----------+------------+
| 0           | Y        | N          |
| 1           | Y        | Y          |
| 2           | N        | Y          |
| 3           | Y        | Y          |
| 4           | N        | N          |
+-------------+----------+------------+
Output: 
+-------------+
| product_id  |
+-------------+
| 1           |
| 3           |
+-------------+

Write a solution to find the ids of products that are both low fat and recyclable.


'''

In [13]:
spark.version

'3.5.7'

In [6]:
Data=[(0,'Y','N'),(1,'Y','Y'),\
     (3,'Y','Y'),(2,'N','Y'),\
     (4,'N','N')]
Schema=StructType([StructField("Product_Id",IntegerType(),True),\
                   StructField("low_fats",StringType(),True),\
                              StructField("recyclable",StringType(),True)] 
)
Products = spark.createDataFrame(Data,Schema)

In [10]:
#Products.show()
Products.filter((col("low_fats") == "Y") & (col("recyclable") == "Y")).show()

+----------+--------+----------+
|Product_Id|low_fats|recyclable|
+----------+--------+----------+
|         1|       Y|         Y|
|         3|       Y|         Y|
+----------+--------+----------+



### 584.Find Customer Refree

''' 


Table: Customer

+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| name        | varchar |
| referee_id  | int     |
+-------------+---------+
In SQL, id is the primary key column for this table.
Each row of this table indicates the id of a customer, their name, and the id of the customer who referred them.
 

Find the names of the customer that are either:

referred by any customer with id != 2.
not referred by any customer.
Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: 
Customer table:
+----+------+------------+
| id | name | referee_id |
+----+------+------------+
| 1  | Will | null       |
| 2  | Jane | null       |
| 3  | Alex | 2          |
| 4  | Bill | null       |
| 5  | Zack | 1          |
| 6  | Mark | 2          |
+----+------+------------+
Output: 
+------+
| name |
+------+
| Will |
| Jane |
| Bill |
| Zack |
+------+


'''

In [14]:
Data=[(1,"Will",None),\
      (2,'Jane',None),\
      (3,'Alex',2),\
      (4,'Bill',None),\
      (5,'Zack',1),\
      (6,'Mark',2)]

Schema=StructType([StructField("id",IntegerType(),True),\
                  StructField("name",StringType(),True),\
                  StructField("refree_id",IntegerType(),True)])

Customer= spark.createDataFrame(Data,Schema)

In [15]:
Customer.show()

+---+----+---------+
| id|name|refree_id|
+---+----+---------+
|  1|Will|     NULL|
|  2|Jane|     NULL|
|  3|Alex|        2|
|  4|Bill|     NULL|
|  5|Zack|        1|
|  6|Mark|        2|
+---+----+---------+



In [17]:
Customer.filter((col("refree_id")!=2) | (col("refree_id").isNull())).select("name").show()

+----+
|name|
+----+
|Will|
|Jane|
|Bill|
|Zack|
+----+



### 595. Big Countries

'''

Table: World

+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| name        | varchar |
| continent   | varchar |
| area        | int     |
| population  | int     |
| gdp         | bigint  |
+-------------+---------+
name is the primary key (column with unique values) for this table.
Each row of this table gives information about the name of a country, the continent to which it belongs, its area, the population, and its GDP value.
 

A country is big if:

it has an area of at least three million (i.e., 3000000 km2), or
it has a population of at least twenty-five million (i.e., 25000000).
Write a solution to find the name, population, and area of the big countries.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: 
World table:
+-------------+-----------+---------+------------+--------------+
| name        | continent | area    | population | gdp          |
+-------------+-----------+---------+------------+--------------+
| Afghanistan | Asia      | 652230  | 25500100   | 20343000000  |
| Albania     | Europe    | 28748   | 2831741    | 12960000000  |
| Algeria     | Africa    | 2381741 | 37100000   | 188681000000 |
| Andorra     | Europe    | 468     | 78115      | 3712000000   |
| Angola      | Africa    | 1246700 | 20609294   | 100990000000 |
+-------------+-----------+---------+------------+--------------+
Output: 
+-------------+------------+---------+
| name        | population | area    |
+-------------+------------+---------+
| Afghanistan | 25500100   | 652230  |
| Algeria     | 37100000   | 2381741 |
+-------------+------------+---------+


'''

In [23]:
Data = [['Afghanistan', 'Asia', 652230, 25500100, 20343000000], ['Albania', 'Europe', 28748, 2831741, 12960000000],
        ['Algeria', 'Africa', 2381741, 37100000, 188681000000], ['Andorra', 'Europe', 468, 78115, 3712000000],
        ['Angola', 'Africa', 1246700, 20609294, 100990000000]]

Schema=StructType([StructField("name",StringType(),True),\
                  StructField("continent",StringType(),True),\
                  StructField("area",IntegerType(),True),\
                  StructField("population",IntegerType(),True),\
                   StructField("gdp",LongType(),True)])

World = spark.createDataFrame(Data,Schema)

In [24]:
#World.show()
World.filter((col("area")>=3000000) | (col("populatio"))).select().show()

+-----------+---------+-------+----------+------------+
|       name|continent|   area|population|         gdp|
+-----------+---------+-------+----------+------------+
|Afghanistan|     Asia| 652230|  25500100| 20343000000|
|    Albania|   Europe|  28748|   2831741| 12960000000|
|    Algeria|   Africa|2381741|  37100000|188681000000|
|    Andorra|   Europe|    468|     78115|  3712000000|
|     Angola|   Africa|1246700|  20609294|100990000000|
+-----------+---------+-------+----------+------------+

